In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report
)
import torch
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer
from tqdm.notebook import tqdm
from experiment import preprocess_text

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.precision', 3)

In [2]:
DATA_PATH = Path('raw/lith_dataset_wide_full.csv')

In [3]:
MODEL_ID = "knowledgator/opir-multitask-multilang-v1.0"
model = GLiClassModel.from_pretrained(MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

Loading weights:   0%|          | 0/207 [00:00<?, ?it/s]

[transformers] You are using a model of type `GLiClass` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


## Zero-shot binary hate/not-hate classifier

In [4]:
binary_classifier = ZeroShotClassificationPipeline(
    model=model,
    tokenizer=tokenizer,
    classification_type="single-label",
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

In [5]:
data = pd.read_csv(DATA_PATH)
hate_df = pd.DataFrame(data={
    'text': list(map(preprocess_text, tqdm(data['text']))), 
    'label': data.drop(labels='text', axis=1).sum(axis=1).astype(int)})
hate_df.head()

  0%|          | 0/9887 [00:00<?, ?it/s]

,text,label
0,"Abu, Izraelis ir Palestina, yra blogai, nė vie...",1
1,Išnaikint pid,1
2,"Jau pastebimos tendencijos, kad LGB bendruomen...",1
3,Jei jie nesireikstu ir tiesiog gyventu niekas ...,1
4,Nes pyderai duchai,1


In [6]:
result = binary_classifier(hate_df['text'].tolist(), labels=["not-hate", "hate"], batch_size=2)

100%|██████████| 4944/4944 [04:25<00:00, 18.63it/s]


In [7]:
predicted = list(map(lambda r: max(r, key=lambda x: x["score"])['label'], result))
predicted = (np.array(predicted) == 'hate').astype(int)
print(classification_report(hate_df['label'], predicted))

              precision    recall  f1-score   support

           0       0.46      0.85      0.60      4185
           1       0.72      0.27      0.39      5702

    accuracy                           0.52      9887
   macro avg       0.59      0.56      0.50      9887
weighted avg       0.61      0.52      0.48      9887



## Few-shot multilabel classifier

In [8]:
multi_classifier = ZeroShotClassificationPipeline(
    model=model,
    tokenizer=tokenizer,
    classification_type="multi-label",
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
N_EXAMPLES = 3

In [ ]:
data = pd.read_csv(DATA_PATH)
data = data[hate_df['label'] == 1]   # Use only hate speen data
LABELS = set(data.columns.tolist()) - {'text'}
reference_labels = {
    'race': ['Ukrainian', 'American', 'Russian', 'Jewish', 'Lithuanian', 'Dark-skinned', 'Other-African', 'Arab', 'Belorussian', 'Unspecified-ethnicity', 'Indian', 'Polish', 'German', 'French', 'Palestinians', 'Roma', 'Mongolian', 'Chinese', 'Central-Asian', 'Korean', 'Swedish', 'Japanese', 'Latvian', 'English', 'Other-Asian', 'Estonian', 'Other', 'Caucasian'], 
    'orientation': ['Gay-Men', 'Orientation-/-Gender-Unspecified', 'Lesbians', 'S.Orientation-Other', 'Transgender-Individuals', 'Pedophile', 'Homosexual-individuals', 'Males', 'Trans-women', 'Females'], 
    'countries': ['Russia', 'USA', 'Lithuania', 'UNIVERSAL', 'Germany', 'Belorus', 'Israel', 'Ukraine', 'Palestine-region', 'Other', 'Other-Asian', 'Poland', 'UK', 'China', 'Caucasian-Countries', 'Other-European', 'Japan'], 
    'political': ['PV_Pro-Russian', 'PP_LSDP', 'PV_Politician', 'PP_DP', 'PP_TS-LKD', 'PP_LS', 'PP_Tautininkai', 'PV_Eponyms', 'PP_Other', 'PP_LVŽS', 'PV_Conservative', 'PV_Left/socialist', 'AC_Political-activists', 'PV_Communist/USSR', 'PP_NS', 'PV_Liberal', 'PV_OTHER', 'PP_LP'], 
    'religion': ['Catholics', 'Muslims-(Islam)', 'Christian', 'General', 'Orthodox-Christians', 'Pagans', 'Atheists', 'Other-Christian', 'Eastern-religions', 'Religious-people', 'Jews']
}
# Remove labels which are not present in the dataset
labels = dict()
for group, label_list in reference_labels.items():
    labels[group] = list(set(label_list).intersection(LABELS))

In [10]:
texts = list(map(preprocess_text, tqdm(data['text'])))
data_groups = dict()
data_groups['text'] = texts
for group, group_labels in labels.items():
    data_groups[group] = data[group_labels].sum(axis=1)
data_groups = pd.DataFrame(data_groups)
multi_df = pd.DataFrame({
    'text': texts,
    'labels': data.drop(labels='text', axis=1).apply(lambda x: [k for k, v in x.to_dict().items() if v > 0], axis=1)
})
examples = list()
for label in LABELS:
    sample = multi_df[multi_df['labels'].apply(lambda x: label in x)]
    if sample.shape[0] > N_EXAMPLES:
        examples.extend(sample[:N_EXAMPLES].to_dict(orient='records'))

  0%|          | 0/5702 [00:00<?, ?it/s]

In [11]:
result = multi_classifier(multi_df['text'].tolist(), labels=labels, batch_size=2, examples=examples)

100%|██████████| 2851/2851 [26:56<00:00,  1.76it/s]


In [12]:
print(result[0])

[{'label': 'race.Ukrainian', 'score': 0.678246259689331}, {'label': 'race.Jewish', 'score': 0.511855959892273}, {'label': 'race.Russian', 'score': 0.8023308515548706}, {'label': 'orientation.Pedophile', 'score': 0.9770772457122803}, {'label': 'countries.Ukraine', 'score': 0.9204075336456299}]


In [14]:
def calculate_scores(actual, predictions, task_name, average='binary', pos_label=1):
    precision, recall, f1, _ = precision_recall_fscore_support(actual, predictions, average=average, zero_division=0, pos_label=pos_label)
    try:
        roc_auc = roc_auc_score(actual, predictions, multi_class='ovr')
    except:
        roc_auc = np.nan
    return {
       'target': task_name,
       'accuracy': accuracy_score(actual, predictions),
       'f1_score': f1,
       'precision': precision,
       'recall': recall,
       'roc_auc': roc_auc
    }

pred_groups = np.zeros((len(result), len(labels)))
pred_detailed = np.zeros((len(result), len(LABELS)))
GROUPS = list(labels.keys())
LABELS = list(LABELS)
for ind, inst_res in enumerate(result):
    for res in inst_res:
        grp, lbl = tuple(res['label'].split('.'))
        if grp in GROUPS:
            pred_groups[ind, GROUPS.index(group)] = 1
        if lbl in LABELS:
            pred_detailed[ind, LABELS.index(lbl)] = 1
pred_detailed = pd.DataFrame(pred_detailed, columns=LABELS)
pred_groups = pd.DataFrame(pred_groups, columns=GROUPS)
metrics_detail = [calculate_scores(data[label], pred_detailed[label], label) for label in LABELS]
metrics_group = [calculate_scores(data_groups[label], pred_groups[label], label) for label in GROUPS]


Print detailed metrics:

In [15]:
print(pd.DataFrame(metrics_detail))

                              target  accuracy  f1_score  precision  recall  roc_auc
0                    PV_Conservative     0.954     0.007      0.013   0.005    0.496
1                      Other-African     0.999     0.000      0.000   0.000    0.500
2                             French     0.999     0.000      0.000   0.000    0.500
3             AC_Political-activists     0.996     0.000      0.000   0.000    0.500
4                            Gay-Men     0.878     0.072      0.587   0.039    0.517
5                              PP_DP     0.990     0.067      0.222   0.039    0.519
6                            Belorus     0.998     0.125      0.167   0.100    0.550
7                            PP_LVŽS     0.998     0.000      0.000   0.000    0.500
8                      PV_Politician     0.949     0.000      0.000   0.000    0.500
9                                USA     0.983     0.059      0.136   0.038    0.517
10                           PP_LSDP     0.980     0.017      1.0

Print metrics for general groups: 

In [17]:
print(pd.DataFrame(metrics_group))

        target  accuracy  f1_score  precision  recall  roc_auc
0         race     0.719     0.000      0.000     0.0      0.5
1  orientation     0.860     0.000      0.000     0.0      0.5
2    countries     0.933     0.000      0.000     0.0      0.5
3    political     0.627     0.000      0.000     0.0      0.5
4     religion     0.134     0.237      0.134     1.0      0.5
